In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window 

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi spark: ", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 08:48:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi spark:  3.5.9


In [8]:
import numpy as np
import pandas as pd

np.random.seed
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]

# Tabel referensi/master: target dan manafer per kategori dengan data relatif statis
data_produk = {
    "kategori": kategori_list,
    "target_bulanan": [50000000, 40000000, 30000000, 25000000, 20000000],
    "manager": ["Andi", "Budi", "Citra", "Dewi", "Eka"],
}
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))

# Tabel transaksi: data yang terus bertambah setiap hari
n = 300
data_transaksi = {
    "order_id": [f"O{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))

print("df_produk: ")
df_produk.show()
print("df_transaksi (5 baris pertama dari total", df_transaksi.count(), "baris):")
df_transaksi.show(5)

df_produk: 


+--------------------+--------------+-------+
|            kategori|target_bulanan|manager|
+--------------------+--------------+-------+
|          Elektronik|      50000000|   Andi|
|             Fashion|      40000000|   Budi|
|   Makanan & Minuman|      30000000|  Citra|
|Kesehatan & Kecan...|      25000000|   Dewi|
|        Rumah Tangga|      20000000|    Eka|
+--------------------+--------------+-------+



df_transaksi (5 baris pertama dari total 300 baris):
+--------+-----------------+--------+----------+
|order_id|         kategori|    kota|pendapatan|
+--------+-----------------+--------+----------+
|      O0|     Rumah Tangga|Semarang|    315232|
|      O1|       Elektronik|Semarang|    318117|
|      O2|          Fashion|Magelang|    424821|
|      O3|Makanan & Minuman|Semarang|    287188|
|      O4|          Fashion|Semarang|    492363|
+--------+-----------------+--------+----------+
only showing top 5 rows



In [10]:
# menggunakan DataFrame API (join), tampilkan seluruh transaksi kota Solo beserta nama manager
# gunakan left join untuk melihat detail transaksi beserta info manager

df_solo = (
    df_transaksi
    .filter(col("kota") == "Solo")
    .join(df_produk, on="kategori", how="left")
)
df_solo.show(5)

+--------------------+--------+----+----------+--------------+-------+
|            kategori|order_id|kota|pendapatan|target_bulanan|manager|
+--------------------+--------+----+----------+--------------+-------+
|Kesehatan & Kecan...|      O7|Solo|    203379|      25000000|   Dewi|
|Kesehatan & Kecan...|     O17|Solo|     74987|      25000000|   Dewi|
|          Elektronik|     O16|Solo|    481238|      50000000|   Andi|
|             Fashion|     O11|Solo|    352117|      40000000|   Budi|
|        Rumah Tangga|      O5|Solo|    348058|      20000000|    Eka|
+--------------------+--------+----+----------+--------------+-------+
only showing top 5 rows



In [12]:
# Menggunakan window function, tentukan transaksi dengan pendapatan tertinggi (peringkat 1 saja) di setiap kota (bukan kategori). tidak boleh menggunakan rank().

window_spec = Window.partitionBy("kota").orderBy(col("pendapatan").desc())

df_top_kota = (
    df_gabung 
    .withColumn("peringkat", row_number().over(window_spec))
    .filter(col("peringkat") == 1)
)

df_top_kota.show()

+--------------------+--------+--------+----------+--------------+-------+---------+
|            kategori|order_id|    kota|pendapatan|target_bulanan|manager|peringkat|
+--------------------+--------+--------+----------+--------------+-------+---------+
|Kesehatan & Kecan...|    O106|Magelang|    489013|      25000000|   Dewi|        1|
|             Fashion|      O4|Semarang|    492363|      40000000|   Budi|        1|
|   Makanan & Minuman|    O222|    Solo|    498664|      30000000|  Citra|        1|
+--------------------+--------+--------+----------+--------------+-------+---------+



In [13]:
df_transaksi.createOrReplaceTempView("transaksi")

spark.sql("""
    SELECT kota,
            ROUND(AVG(pendapatan), 2) AS rata_rata_pendapatan
    FROM transaksi
    GROUP BY kota
    ORDER BY rata_rata_pendapatan DESC
""").show()

[Stage 27:>                                                         (0 + 4) / 4]

+--------+--------------------+
|    kota|rata_rata_pendapatan|
+--------+--------------------+
|    Solo|           301743.59|
|Semarang|           267824.81|
|Magelang|           256909.76|
+--------+--------------------+

